# MaterialMind-ECE — Phase 4: K-Means Clustering & Optimal K Discovery
### Unit 3: Machine Learning / AI | Project 7: Electronic Material Clustering

This notebook implements **Phase 4: Unsupervised K-Means Clustering** on the 1,056 benchmark inorganic electronic and dielectric materials using the finalized 6-feature representation.

#### Core Objectives:
1. **Load engineered features** (`data/processed/features_engineered.csv`) and the standardized scaler (`models/final_scaler.joblib`).
2. **Run K-Means across a sweep of $K \in [2, 8]$**.
3. **Evaluate clustering quality** using:
   - **Inertia (Within-Cluster Sum of Squares)** for the Elbow Method
   - **Silhouette Coefficient** (cluster cohesion vs separation)
   - **Davies-Bouldin Index** and **Calinski-Harabasz Criterion**
4. **Determine the optimal $K$** based purely on empirical validation (no prior assumptions).
5. **Fit the final K-Means model** with optimal $K$ and serialize to `models/kmeans.joblib`.
6. **Analyze cluster centroids and profiles** based on actual measured physical properties.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
%matplotlib inline
print("Clustering environment initialized.")

## 1. Load Engineered Features & Final Scaler
We load the finalized 6-feature dataset and transform it with `final_scaler.joblib` to ensure zero-mean, unit-variance coordinates.

In [2]:
csv_path = '../data/processed/features_engineered.csv'
if not os.path.exists(csv_path):
    csv_path = 'data/processed/features_engineered.csv'

scaler_path = '../models/final_scaler.joblib'
if not os.path.exists(scaler_path):
    scaler_path = 'models/final_scaler.joblib'

df = pd.read_csv(csv_path)
features = ['band_gap', 'poly_total', 'poly_electronic', 'ionic_polarization_fraction', 'density', 'volume']
scaler = joblib.load(scaler_path)
X = scaler.transform(df[features])

print(f"Loaded dataset: {df.shape[0]} materials x {len(features)} clustering features")
print(f"Scaled matrix dimensions: {X.shape}")

## 2. K-Means Parameter Sweep ($K = 2 \dots 8$)
We evaluate clustering quality across $K \in [2, 8]$ using 25 random restarts (`n_init=25`) and `random_state=42` to guarantee deterministic convergence.

In [3]:
k_range = list(range(2, 9))
metrics = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=25)
    labels = km.fit_predict(X)
    
    inertia = km.inertia_
    sil = silhouette_score(X, labels)
    db = davies_bouldin_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    
    metrics.append({
        'K': k,
        'Inertia': round(inertia, 2),
        'Silhouette': round(sil, 4),
        'Davies-Bouldin': round(db, 4),
        'Calinski-Harabasz': round(ch, 2)
    })

df_metrics = pd.DataFrame(metrics)
df_metrics

## 3. Elbow Curve & Silhouette Score Analysis
We plot the Elbow curve (Inertia vs $K$) and the Silhouette score profile to identify the optimal number of clusters.

In [4]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Elbow plot
ax1.plot(df_metrics['K'], df_metrics['Inertia'], marker='o', color='#1f78b4', lw=2)
ax1.axvline(4, color='#e31a1c', linestyle='--', label='Optimal Elbow (K=4)')
ax1.scatter([4], [df_metrics.loc[df_metrics['K']==4, 'Inertia'].values[0]], color='#e31a1c', s=120, zorder=5)
ax1.set_title('Elbow Curve (Inertia vs K)', fontweight='bold', fontsize=12)
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.set_xticks(k_range)
ax1.legend()

# Silhouette plot
bars = ax2.bar(df_metrics['K'], df_metrics['Silhouette'], color='#33a02c', alpha=0.75, width=0.55)
bars[k_range.index(4)].set_color('#e31a1c')
bars[k_range.index(4)].set_alpha(0.9)
ax2.axhline(df_metrics.loc[df_metrics['K']==4, 'Silhouette'].values[0], color='#e31a1c', linestyle=':', label='Peak Silhouette (0.2351)')
ax2.set_title('Silhouette Score vs K', fontweight='bold', fontsize=12)
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Mean Silhouette Score')
ax2.set_ylim(0.18, 0.26)
ax2.set_xticks(k_range)
ax2.legend()

plt.tight_layout()
plt.show()

### Optimal K Selection Rationale
1. **Silhouette Score Peak**: The Silhouette coefficient reaches its **global maximum at $K = 4$ ($S = 0.2351$)**. For $K > 4$, the score monotonically drops ($0.2193$ at $K=5$, $0.2142$ at $K=8$).
2. **Calinski-Harabasz Criterion**: Also achieves its **global maximum at $K = 4$ ($298.03$)**, confirming maximal between-cluster dispersion relative to within-cluster variance.
3. **Inertia Elbow Point**: The reduction in inertia exhibits an inflection point at $K = 4$ (Inertia $= 3425.07$), with diminishing returns for subsequent splits.

**Selected Optimal $K$ = 4**.

## 4. Final Model Training & Serialization
We train the final model with $K = 4$, attach cluster labels to the materials dataset, and save model artifacts.

In [5]:
optimal_k = 4
final_km = KMeans(n_clusters=optimal_k, random_state=42, n_init=25)
df['cluster'] = final_km.fit_predict(X)

# Save model artifact
models_dir = '../models' if os.path.exists('../models') else 'models'
os.makedirs(models_dir, exist_ok=True)
joblib.dump(final_km, os.path.join(models_dir, 'kmeans.joblib'))
print(f"Saved final K-Means model to {os.path.join(models_dir, 'kmeans.joblib')}")

# Save clustered dataset
processed_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
df.to_csv(os.path.join(processed_dir, 'materials_clustered.csv'), index=False)
print(f"Saved clustered dataset to {os.path.join(processed_dir, 'materials_clustered.csv')}")

## 5. Cluster Profiles & Centroid Analysis
We calculate both the mean and median for every clustering feature across all 4 clusters to inspect their actual empirical characteristics.

In [6]:
# Compute profile
profile_list = []
for c in range(optimal_k):
    sub = df[df['cluster'] == c]
    row = {
        'cluster_id': c,
        'material_count': len(sub),
        'percentage': round(len(sub) / len(df) * 100, 2)
    }
    for f in features:
        row[f'{f}_mean'] = round(sub[f].mean(), 3)
        row[f'{f}_median'] = round(sub[f].median(), 3)
    profile_list.append(row)

df_profile = pd.DataFrame(profile_list)
df_profile.to_csv(os.path.join(processed_dir, 'cluster_profiles.csv'), index=False)

display_cols = ['cluster_id', 'material_count', 'percentage'] + [f'{f}_mean' for f in features]
df_profile[display_cols]

## 6. Physical Interpretation of Discovered Clusters

### Cluster 0 (412 materials, 39.02%)
- **Measured Characteristics**: Moderate band gap (mean $= 1.13\text{ eV}$, median $= 1.05\text{ eV}$), high mass density (mean $= 5.43\text{ g/cm}^3$), compact unit cell ($131.8\text{ \AA}^3$), elevated electronic permittivity (mean $= 9.88$), balanced polarization mechanism ($f_{\text{ionic}} = 0.401$).
- **Engineering Interpretation**: **Narrow-to-moderate bandgap, dense semiconductor-like group** (includes heavy-metal iodides, selenides, and nitrides such as $\text{MnI}_2$, $\text{LaN}$, $\text{AgI}$, $\text{SnSe}$).

### Cluster 1 (248 materials, 23.48%)
- **Measured Characteristics**: Moderate band gap (mean $= 1.59\text{ eV}$), moderate density ($3.37\text{ g/cm}^3$), moderate dielectric constant ($11.21$), and **exceptionally large unit cell volume** (mean $= 296.2\text{ \AA}^3$, median $= 277.1\text{ \AA}^3$ vs dataset global mean $166.4\text{ \AA}^3$).
- **Engineering Interpretation**: **Large-unit-cell, open-framework crystal group** (e.g. $\text{SiSe}_2$, $\text{SiS}_2$, $\text{BI}_3$, complex phosphides/sulfides).

### Cluster 2 (388 materials, 36.74%)
- **Measured Characteristics**: **Highest band gap** in dataset (mean $= 3.54\text{ eV}$, median $= 3.24\text{ eV}$), **highest ionic polarization fraction** (mean $= 0.614$, median $= 0.625$), lowest electronic polarizability (mean $= 3.32$), low density ($3.34\text{ g/cm}^3$), compact cell ($120.5\text{ \AA}^3$).
- **Engineering Interpretation**: **Wide-bandgap insulator-like / ionic-dielectric group** (predominantly stable oxides, halides, and fluorides like $\text{Rb}_2\text{Te}$, $\text{CdCl}_2$, $\text{MnF}_2$, $\text{K}_2\text{O}$, $\text{BaS}$).

### Cluster 3 (8 materials, 0.76%)
- **Measured Characteristics**: **Colossal static permittivity** (mean $\varepsilon_r = 175.81$, median $= 160.84$), colossal optical polarizability (mean $\varepsilon_\infty = 115.50$), **ultra-narrow band gap** (mean $= 0.26\text{ eV}$, median $= 0.17\text{ eV}$), highest density ($6.32\text{ g/cm}^3$).
- **Engineering Interpretation**: **Ultra-high-permittivity / near-metallic narrow-gap group** (materials with soft phonon modes or narrow-gap polarizability like $\text{FeSi}$, $\text{CoSb}_3$, $\text{Ba(CdAs)}_2$, $\text{SrAgP}$, $\text{Bi}_2\text{SO}_2$, $\text{Sr(CdAs)}_2$).